# Lab 6, Part 2 — Using pretrained Transformers (Hugging Face)

This notebook covers the three hands-on tasks from the handout. Each one uses a different kind of Transformer:

| Task | Architecture | Model |
|---|---|---|
| Machine translation (EN→FR) | **encoder–decoder** | `Helsinki-NLP/opus-mt-en-fr` (MarianMT) |
| Sentiment analysis, plus semantic search | **encoder-only** | `distilbert-base-uncased-finetuned-sst-2-english`, `sentence-transformers/all-MiniLM-L6-v2` |
| Text generation (GPT-style) | **decoder-only** | `gpt2`, `Qwen/Qwen2.5-0.5B-Instruct` |

**Run this notebook on Google Colab or locally with internet access.** The models are downloaded from the Hugging Face Hub. All of them are small (under 1 GB), so a CPU runtime is enough and a GPU just makes it faster.

The models are loaded with the `Auto*` classes rather than `pipeline(...)`. That makes each architectural piece visible: tokeniser → model → logits → decoding.

In [ ]:
# Colab: run once
%pip install -q "transformers>=4.45" sentencepiece sacremoses torch

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import (AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM,
                          AutoModelForSequenceClassification, AutoModelForCausalLM)

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. Machine translation with an encoder–decoder Transformer

This is the use case the Transformer was originally introduced for. The **encoder** reads the whole English sentence with bidirectional self-attention. The **decoder** generates French one token at a time, using **masked** self-attention over the French produced so far and **cross-attention** to the encoder output.

In [ ]:
mt_name = "Helsinki-NLP/opus-mt-en-fr"
mt_tok = AutoTokenizer.from_pretrained(mt_name)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(mt_name, attn_implementation="eager").to(device).eval()

cfg = mt_model.config
print(f"encoder layers={cfg.encoder_layers}  decoder layers={cfg.decoder_layers}  "
      f"d_model={cfg.d_model}  heads={cfg.encoder_attention_heads}")
print("A decoder layer contains cross-attention:", mt_model.model.decoder.layers[0].encoder_attn.__class__.__name__)

In [ ]:
sentences = [
    "I love machine learning.",
    "The cat is sleeping on the sofa.",
    "Transformers were originally introduced for machine translation.",
    "The bank will close early today, so I will sit on the river bank instead.",
]

def translate(texts, num_beams=4):
    batch = mt_tok(texts, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        out = mt_model.generate(**batch, num_beams=num_beams, max_new_tokens=60)
    return mt_tok.batch_decode(out, skip_special_tokens=True)

for en, fr in zip(sentences, translate(sentences)):
    print(f"EN: {en}\nFR: {fr}\n")

### Looking inside: cross-attention acts as a soft word alignment

The next cell feeds the source sentence and the generated translation back in, and plots the last decoder layer's cross-attention, averaged over heads. Each row is a French token, and it shows which English tokens that row attended to.

In [ ]:
src = "I love machine learning."
enc = mt_tok(src, return_tensors="pt").to(device)
with torch.no_grad():
    gen = mt_model.generate(**enc, num_beams=1, max_new_tokens=30)
    out = mt_model(**enc, decoder_input_ids=gen, output_attentions=True)

cross = out.cross_attentions[-1][0].mean(0).cpu()          # (tgt_len, src_len)
src_toks = mt_tok.convert_ids_to_tokens(enc["input_ids"][0])
tgt_toks = mt_tok.convert_ids_to_tokens(gen[0])

plt.figure(figsize=(6, 5))
plt.imshow(cross, cmap="viridis")
plt.xticks(range(len(src_toks)), src_toks, rotation=45)
plt.yticks(range(len(tgt_toks)), tgt_toks)
plt.xlabel("English (encoder: keys/values)"); plt.ylabel("French (decoder: queries)")
plt.title("Cross-attention, last decoder layer"); plt.colorbar(); plt.tight_layout(); plt.show()

## 2. Encoder-only models: sentiment analysis

An encoder produces a contextual vector for every token. A classification head placed on the first (`[CLS]`) token gives the sentiment. There is no generation, only *understanding* of the existing text.

In [ ]:
sa_name = "distilbert-base-uncased-finetuned-sst-2-english"
sa_tok = AutoTokenizer.from_pretrained(sa_name)
sa_model = AutoModelForSequenceClassification.from_pretrained(sa_name).to(device).eval()
print("labels:", sa_model.config.id2label)

reviews = [
    "This lab was fantastic, I finally understand attention!",
    "The installation failed three times and the documentation was useless.",
    "The movie was not bad at all.",            # negation
    "Oh great, another bug. Just what I needed.",  # sarcasm: a known hard case
    "The package arrived on Tuesday.",           # neutral: the model has no neutral class
]
batch = sa_tok(reviews, return_tensors="pt", padding=True, truncation=True).to(device)
with torch.no_grad():
    probs = sa_model(**batch).logits.softmax(-1).cpu()
for r, p in zip(reviews, probs):
    label = sa_model.config.id2label[int(p.argmax())]
    print(f"{label:8s} {p.max():.3f}  | {r}")

Things to look for in the output:

* **Negation.** "not bad at all" should come out POSITIVE. Bidirectional context lets *not* modify *bad*.
* **Sarcasm.** This often fails, because the surface words are positive.
* **Neutral text.** SST-2 has only two classes, so the model is *forced* to choose one, often with high confidence. A confident output is not the same as a well-calibrated one.

### Bonus: semantic search with an encoder (from the handout's list of encoder-only use cases)

Sentences are embedded by mean-pooling the encoder's token vectors. Documents are then ranked by cosine similarity to the query. Matching is by meaning rather than by exact words.

In [ ]:
emb_name = "sentence-transformers/all-MiniLM-L6-v2"
emb_tok = AutoTokenizer.from_pretrained(emb_name)
emb_model = AutoModel.from_pretrained(emb_name).to(device).eval()

def embed(texts):
    b = emb_tok(texts, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        h = emb_model(**b).last_hidden_state
    m = b["attention_mask"].unsqueeze(-1).float()
    return F.normalize((h * m).sum(1) / m.sum(1), dim=-1)     # mean pooling + L2 norm

docs = [
    "How to reset your account password",
    "Refund policy for damaged items",
    "Shipping times for international orders",
    "Troubleshooting login problems",
    "Recipe for chocolate chip cookies",
]
query = "I can't sign in to my account"
scores = (embed([query]) @ embed(docs).T)[0].cpu()
for s, d in sorted(zip(scores.tolist(), docs), reverse=True):
    print(f"{s:.3f}  {d}")
# Note: the top results share no keyword with the query ("sign in" vs "login"/"password").


## 3. Decoder-only models: GPT-style text generation

A decoder-only model uses **masked** self-attention, so it predicts token $t+1$ from tokens $1..t$. Generation repeats this step: predict a distribution, pick a token, append it, predict again.

First, a check that ties this back to Part 1. In a trained GPT-2, changing a **later** token must leave the logits at **earlier** positions unchanged, because of the causal mask.

In [ ]:
g_tok = AutoTokenizer.from_pretrained("gpt2")
g_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()

a = g_tok("The Transformer was introduced for machine translation", return_tensors="pt").input_ids.to(device)
b = a.clone(); b[0, -1] = g_tok(" cooking").input_ids[0]          # change only the LAST token
with torch.no_grad():
    la, lb = g_model(a).logits, g_model(b).logits
print("earlier positions unchanged:", torch.allclose(la[0, :-1], lb[0, :-1], atol=1e-4))
print("last position changed:     ", not torch.allclose(la[0, -1], lb[0, -1], atol=1e-4))

In [ ]:
prompt = "Machine learning is"
ids = g_tok(prompt, return_tensors="pt").input_ids.to(device)

# Next-token distribution: next-token prediction is classification over ~50k tokens
with torch.no_grad():
    p = g_model(ids).logits[0, -1].softmax(-1)
top = p.topk(8)
print("Top next tokens after", repr(prompt))
for prob, i in zip(top.values, top.indices):
    print(f"  {g_tok.decode(int(i))!r:15} {prob:.3f}")

def generate(**kw):
    torch.manual_seed(0)
    with torch.no_grad():
        out = g_model.generate(ids, max_new_tokens=40, pad_token_id=g_tok.eos_token_id, **kw)
    return g_tok.decode(out[0], skip_special_tokens=True)

print("\n[greedy]\n", generate(do_sample=False))
print("\n[sampling, T=0.8, top_p=0.9]\n", generate(do_sample=True, temperature=0.8, top_p=0.9))
print("\n[sampling, T=1.5]\n", generate(do_sample=True, temperature=1.5))

Greedy decoding with base GPT-2 usually falls into **repetition loops**. Sampling at a moderate temperature is more varied, and at a high temperature it becomes incoherent. GPT-2 is a *base* model: it continues text and does not follow instructions.

### An instruction-tuned decoder (chat model)

A small instruction-tuned model responds to requests because of its post-training. The architecture is the same kind of decoder-only stack. A **chat template** wraps each message in special role tokens before generation.

In [ ]:
chat_name = "Qwen/Qwen2.5-0.5B-Instruct"
c_tok = AutoTokenizer.from_pretrained(chat_name)
c_model = AutoModelForCausalLM.from_pretrained(chat_name).to(device).eval()

def chat(user_msg, max_new_tokens=200):
    msgs = [{"role": "user", "content": user_msg}]
    text = c_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = c_tok(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = c_model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    return c_tok.decode(out[0, ids.input_ids.shape[1]:], skip_special_tokens=True)

print(repr(c_tok.apply_chat_template([{"role": "user", "content": "Hi"}], tokenize=False,
                                     add_generation_prompt=True)))   # see the special tokens
for q in ["Explain self-attention in two sentences.",
          "Write a Python function that checks whether a string is a palindrome.",
          "Summarise: The Transformer replaced recurrence with attention, allowing parallel training "
          "and better long-range dependencies, and it became the basis of modern LLMs."]:
    print(f"\n### {q}\n{chat(q)}")

## Summary

| | Encoder–decoder (MarianMT) | Encoder-only (DistilBERT, MiniLM) | Decoder-only (GPT-2, Qwen) |
|---|---|---|---|
| Attention | encoder: bidirectional; decoder: causal + cross | bidirectional | causal only |
| Output | a new sequence, conditioned on the input | labels or embeddings | a continuation of the input |
| Used here for | EN→FR translation | sentiment, semantic search | free text generation, chat |

Observations to record when you run it:

* the translation quality on the ambiguous "bank" sentence;
* the cross-attention alignment for "I love machine learning";
* the negation, sarcasm and neutral failure cases in sentiment;
* the causal-mask check on GPT-2;
* how greedy decoding and different temperatures change the output;
* base vs instruction-tuned behaviour.